# Piastra — interactive driver

This notebook mirrors `main.py`: edit the parameter cell, then run the cells top to bottom. Each run sets up a grid, loads a test problem, builds the matching solver, and integrates in time with live plotting.


## Modes and test problems

Pass `mode` and `problem` to `Parameters`. `'user_defined'` exists for every mode. The authoritative name -> initial-condition mapping is in `src/misc/helpers.py`.

| Mode | Problems |
|------|----------|
| `adv`  | smooth1D, disc1D, smooth2D, disc2D |
| `HD`   | sod1Dcart, sod1Dcyl, sod1Dsph, strong1D, DBW1D, shuosher1D, einfeldt1D, sod2Dcart, sod2Dsph, sod2Dpol, sedov2Dcart, sedov2Dcyl, RP2D, gresho2D, KHI2D, RTI2D, shock-cloud, gap-opening, jet2Dcyl |
| `rHD`  | RP1, RP3, RP4, RP5, RP2D, RTI, jet2Dcart, jet2Dcyl |
| `MHD`  | BW1D, toth1D, RJ1D, alfven1D, blast2Dcart, blast2Dcyl, blast2Dsph, rotor2D, OT2D, current-sheet, field-loop, disk2D, shock-cloud |
| `rMHD` | BW1D, RP2, RP3, RP4, blast2D, rotor2D |
| `SWE`  | dam1D, bump1D, bathtub2D, expl2D, tsunami2D, ocean2D, atmo2D, dam2D, jet2D, KHI2D |
| `diff` | gauss1D, gauss2D, step1D, sine1D, cross2D, ring2D, cyl2D |

**Solvers** — `adv`: adv, LW · `HD`: LLF, HLL, HLLC, Roe, Exact · `rHD`: LLF, HLL, HLLC · `MHD`: LLF, HLL, HLLC, HLLD (`divb_tr`: CT, GLM, 8wave) · `rMHD`: LLF, HLL (`divb_tr`: CT) · `SWE`: LLF, HLL, Exact · `diff`: expl, rkl2 (`rkl2_stages` >= 2).

**Reconstruction** — PCM, PLM, PPMorig, PPM, WENO, MP5.  **Time integration** — RK1, RK2, RK3.


## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.grid.grid_setup import Grid
from src.sim_state import SimState
from src.parameters import Parameters
from src.models.MHD.MHD_step_CT import MHD2D_CT
from src.models.MHD.MHD_step_8wave import MHD2D_8wave
from src.models.MHD.MHD_step_GLM import MHD2D_GLM
from src.models.HD.HD_step import HD2D
from src.models.rHD.rHD_step import rHD2D
from src.models.rMHD.rMHD_step import rMHD2D_CT
from src.models.adv.adv_step import Adv2D
from src.models.SWE.SWE_step import SWE2D
from src.models.diff.diff_step import Diff2D
from src.misc.helpers import run_simulation, initial_model
from src.misc.io_visual import plot_setup, plotting

## Solver dispatch

Maps a mode string to a callable that builds the solver. For MHD the divergence-control scheme is resolved from `par.divb_tr` (CT / GLM / 8wave).


In [ ]:
SOLVER_DISPATCH = {
    "adv":  lambda grid, state, eos, par: Adv2D(grid, state, par),
    "SWE":  lambda grid, state, eos, par: SWE2D(grid, state, par),
    "HD":   lambda grid, state, eos, par: HD2D(grid, state, eos, par),
    "rHD":  lambda grid, state, eos, par: rHD2D(grid, state, eos, par),
    "MHD":  lambda grid, state, eos, par: (
        MHD2D_CT(grid, state, eos, par) if par.divb_tr == "CT" else
        MHD2D_GLM(grid, state, eos, par) if par.divb_tr == "GLM" else
        MHD2D_8wave(grid, state, eos, par)),
    "rMHD": lambda grid, state, eos, par: rMHD2D_CT(grid, state, eos, par),
    "diff": lambda grid, state, eos, par: Diff2D(grid, state, par),
}

## Define simulation parameters

Edit this cell and re-run the notebook to change the simulation.


In [ ]:
par = Parameters(
    mode="HD",
    Nx1=64,
    Nx2=64,
    problem="KHI2D",
    solver_type='HLLC',
    # timestep
    CFL=0.7,
    rec_type='PPM',
    RK_order='RK3',
)

print(par)  # show setup

## Build grid, allocate state, load the initial condition

`initial_model` sets the grid geometry, primitive variables, boundary conditions, final time, and equation of state for the chosen problem.


In [ ]:
grid = Grid(par.Nx1, par.Nx2, par.Ngc)

# Unified state container for all modes
state = SimState(grid, par)

grid, state, par, eos = initial_model(grid, state, par)

## Select the solver

In [ ]:
solver = SOLVER_DISPATCH[par.mode](grid, state, eos, par)

## Choose the variable to visualise

Temperature for diffusion, water height for shallow water, density otherwise.


In [ ]:
if par.mode == "diff":
    var_to_plot = state.T
elif par.mode == "SWE":
    var_to_plot = state.h
else:
    var_to_plot = state.dens

## Run the simulation

`run_simulation` marches in time until `par.timefin`, refreshing the plot every `nsteps_visual` steps, and returns the final state and time.


In [ ]:
nsteps_visual = 40
state, par.timenow = run_simulation(
    grid, state, par, solver, var_to_plot, nsteps_visual
)

## Optional: magnetic-field divergence diagnostic (MHD / rMHD, 2D)

A clean scheme keeps div(B) near round-off; this is a quick visual check.


In [ ]:
if (par.mode == "MHD" or par.mode == "rMHD") and (par.Nx1 > 1) and (par.Nx2 > 1):
    divB = np.zeros(grid.grid_shape, dtype=np.double)
    divB[grid.Ngc:grid.Nx1r, grid.Ngc:grid.Nx2r] = state.divB
    line, ax, fig, im = plot_setup(grid, divB, par.timenow)
    # plotting(grid, divB, par.timenow, line, ax, fig, im)

---
**Author:** mrkondratyev